In [ ]:
import torch
from torch import optim, nn
import torchvision
from torchvision import datasets, models, transforms
import numpy as np
from torch.utils.data.sampler import SubsetRandomSampler
import os
import pandas as pd
from PIL import Image 
import albumentations as A

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:

model = models.resnet18(weights='IMAGENET1K_V1')
model.eval()

for name, child in model.named_children():
  print(name)

In [ ]:
train_dir = 'phone_or/train'
valid_dir = 'phone_or/test'

In [ ]:
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def visualize(image):
    plt.figure(figsize=(10, 10))
    plt.axis('off')
    plt.imshow(image)

image = cv2.imread(train_dir + '/phone/phones(1).png')
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
visualize(image)

In [ ]:
tt = A.ShiftScaleRotate(p=0.85)
aug = tt(image=image)

augmented_image = aug['image']
visualize(augmented_image)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

import matplotlib as mpl
mpl.rcParams['axes.grid'] = False
mpl.rcParams['image.interpolation'] = 'nearest'
mpl.rcParams['figure.figsize'] = 35, 25

def show_dataset(dataset, n=8):
  img = np.vstack((np.hstack((np.asarray(dataset[i][0]) for _ in range(n)))
                   for i in range(len(dataset))))
  plt.imshow(img)
  plt.axis('off')

In [ ]:
def load_model(path, model, optimizer):

  checkpoint = torch.load(path)
  model.load_state_dict(checkpoint['model_state_dict'])
  optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
  return model, optimizer


In [ ]:
import torch
from torch import nn
from torch.optim import lr_scheduler
from tqdm.auto import tqdm
import numpy as np


def train_and_test(
    model,
    trainloader,
    validloader,
    epochs,
    path,
    patience=5,      
    min_delta=0.0    
):

    device = next(model.parameters()).device

    criterion = nn.NLLLoss()
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.001
    )
    scheduler = lr_scheduler.StepLR(optimizer, step_size=6, gamma=0.1)

    train_losses, val_losses = [], []
    best_val_loss = np.inf
    epochs_no_improve = 0

    for epoch in range(epochs):

        # ---------------- TRAIN ----------------
        model.train()
        running_loss = 0.0
        train_pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs} [train]")

        for images, labels in train_pbar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            train_pbar.set_postfix(loss=loss.item())

        avg_train_loss = running_loss / len(trainloader)
        train_losses.append(avg_train_loss)

        # ---------------- VALIDATION ----------------
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            val_pbar = tqdm(validloader, desc=f"Epoch {epoch+1}/{epochs} [val]")

            for images, labels in val_pbar:
                images, labels = images.to(device), labels.to(device)

                logps = model(images)
                loss = criterion(logps, labels)
                val_loss += loss.item()

                ps = torch.exp(logps)
                _, preds = ps.topk(1, dim=1)

                preds = preds.squeeze()
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_val_loss = val_loss / len(validloader)
        val_accuracy = correct / total
        val_losses.append(avg_val_loss)

        scheduler.step()

        print(f"\nEpoch {epoch+1}/{epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Acc: {val_accuracy:.4f}")

        # ---------------- EARLY STOPPING ----------------
        if avg_val_loss < best_val_loss - min_delta:
            print(f"Validation loss improved {best_val_loss:.4f} → {avg_val_loss:.4f}. Saving model.")
            best_val_loss = avg_val_loss
            epochs_no_improve = 0

            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
            }, path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("\n⚠ Early stopping triggered. Training stopped.")
                break

    return train_losses, val_losses


In [ ]:
import numpy as np
from torch.utils.data import DataLoader, SubsetRandomSampler
import matplotlib.pyplot as plt


def train_model(model, dataset, epochs=10, batch_size=16, valid_size=0.2, save_path="./model.pt"):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    num_samples = len(dataset)
    indices = np.random.permutation(num_samples)

    split = int(num_samples * valid_size)
    val_idx, train_idx = indices[:split], indices[split:]

    train_loader = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx))
    valid_loader = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx))

    train_losses, val_losses = train_and_test(
        model=model,
        trainloader=train_loader,
        validloader=valid_loader,
        epochs=epochs,
        path=save_path,
        patience=5,
        min_delta=0.0
    )

    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label="Training loss")
    plt.plot(val_losses, label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()
    plt.show()

    print("Training finished. Best model saved to:", save_path)

    return model


In [ ]:
def test_model(model, test_data, batch_size=16):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    testloader = DataLoader(test_data, batch_size=batch_size)

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    print(f"Test Accuracy: {acc:.4f}")

    return acc


In [ ]:
from torch.optim import lr_scheduler

def get_model(device):
  model = models.resnet18(weights='IMAGENET1K_V1') 

  hidd_size = model.fc.in_features // 3
  model.fc = nn.Sequential(nn.Linear(model.fc.in_features, hidd_size), nn.ReLU(), nn.Linear(hidd_size, 2),
                           nn.LogSoftmax(dim=1))    

    
  for name, child in model.named_children():
    if name in ['layer3', 'layer4', 'fc']:
      print(name + ' is unfrozen')
      for param in child.parameters():
        param.requires_grad = True
    else:
      print(name + ' is frozen')
      for param in child.parameters():
        param.requires_grad = False

  model.to(device)
  return model

In [ ]:
import torchshow as ts

test_transforms = transforms.Compose([transforms.Resize(256),
                                      transforms.CenterCrop(224),
                                      transforms.ToTensor(),
                                      transforms.Normalize([0.485, 0.456, 0.406],
                                                           [0.229, 0.224, 0.225])])

test_data = datasets.ImageFolder(valid_dir, transform=test_transforms)
testloader = torch.utils.data.DataLoader(test_data, batch_size=16)
images, labels = next(iter(testloader))
ts.show(images)
print(f"testing examples contain : {len(test_data)}")

In [ ]:
tr = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224,224)),
    torchvision.transforms.ColorJitter(hue=.05, saturation=.05),
    torchvision.transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])]
)

td = datasets.ImageFolder(train_dir, transform=tr)
tl = torch.utils.data.DataLoader(td, batch_size=32)

for images, labels in tl:
  ts.show(images)

print(f"training examples contain: {len(td)}")

In [ ]:
model = get_model(device)
train_data = torchvision.datasets.ImageFolder(train_dir, transform=tr)
train_model(model, train_data)

In [ ]:
test_model(model, test_data)

In [ ]:
df=pd.DataFrame()

tr_list = []
desc_list = []
acc_list = []

transforms0 = torchvision.transforms.Compose([
  torchvision.transforms.ColorJitter(hue=.05, saturation=.05),
  torchvision.transforms.Resize((250,250)),
  torchvision.transforms.CenterCrop(224),
  torchvision.transforms.ToTensor(),
  torchvision.transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])]
)
tr_list.append(transforms0)
desc_list.append("torchvision augs: colorJitter >> do color changes to more attention to shape")

transforms1 = torchvision.transforms.Compose([
  torchvision.transforms.RandomAffine(degrees=40, scale=(.9, 1.1), shear=0),
  torchvision.transforms.RandomPerspective(distortion_scale=0.2),
  torchvision.transforms.RandomHorizontalFlip(),
  torchvision.transforms.Resize((250,250)),
  torchvision.transforms.CenterCrop(224),
  torchvision.transforms.ToTensor(),
  torchvision.transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])]
)
tr_list.append(transforms1)
desc_list.append("torchvision augs: RandomAffine + RandomPerspective >> do some geometry augs")

import torchvision.transforms as T
tt2 = T.Compose([T.ToTensor(), T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0, inplace=False),
                 T.Resize((250,250)), T.CenterCrop(224),
                 T.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])])
tr_list.append(tt2)
desc_list.append("torchvision augs - The RandomErasing() transform randomly selects a rectangular region in an input image and erases its pixels")


In [ ]:
for i, tr in enumerate(tr_list):
  print(">>>>>>", desc_list[i])
  model = get_model(device)
  train_data = torchvision.datasets.ImageFolder(train_dir, transform=tr)
  train_model(model, train_data)
  acc_list.append(test_model(model, test_data))

In [ ]:
df['descr'] = desc_list
df['accuracy'] = acc_list

df.head(3)